# FULL ADJUSTABLE TRANSIT TOOL

### (fill in necessary links/values and run all)

In [ ]:
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import HeatMap
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
import sys
import math
from sklearn.linear_model import LinearRegression
sys.path.append('../..')
#from config import CENSUS_KEY
warnings.filterwarnings('ignore')
MODEL_PATH  = '../../models/xgb_model_2.json' 
CENSUS_KEY = ''
CENSUS_VARIABLES = [
    'B01003_001E',  # total population
    'B08201_001E',  # total households
    'B08201_002E',  # zero-vehicle households
    'B08201_003E',  # one-vehicle households
    'B08301_001E',  # total commuters
    'B08301_002E',  # drove alone
    'B08301_004E',  # carpooled
    'B08301_010E',  # public transit
    'B23025_001E',  # population 16+ (labor force universe)
    'B23025_004E',  # employed civilians
    'B25003_001E',  # total occupied housing units
    'B25003_003E',  # renter-occupied
    'B19013_001E',  # median household income
]

COUNTIES = [
    #Add county FIPS codes here (last 3 digits of the 5-digit code)
    # '001',  # Adams County
    # '005',  # Arapahoe County
    # '013',  # Boulder County
    # '014',  # Broomfield County
    # '059',  # Jefferson County
    # '031',  # Denver County

    # '001',  # Alameda
    # '075',  # San Francisco
    # '081',  # San Mateo
    # '085',  # Santa Clara
    # '013',  # Contra Costa
]
GRID_SPACING_KM  = 0.5       # distance between candidate grid points
N_STATIONS       = 15        # station budget
MIN_SPACING_KM   = 3.0       # minimum distance between any two selected stations

MIN_JOB_KM = 2.0 #minimum distance between job centers

# BOUND BOX AROUND AREA OF INTEREST (min_lon, min_lat, max_lon, max_lat)
BOUNDS = ()

# BOUNDS = (-123.55, 36.85, -121.20, 38.85) 
# for Silicon Valley
# BOUNDS = (-122.75, 37.15, -121.55, 38.05) 
# for San Francisco Bay Area
# BOUNDS = (-105.35, 39.55, -104.60, 40.25) 
# for Denver Metro Area
#BOUNDS = (-119.240929, 33.355132, -117.794815, 34.448494) 
# for Los Angeles Metro Area (LA, Orange, Ventura counties

BOUND_AVG = ((BOUNDS[0] + BOUNDS[2]) / 2, (BOUNDS[1] + BOUNDS[3]) / 2) # average lat/lon for centering maps


STATE_CODE = '08' # 2-digit state FIPS code (06 for California, 08 for Colorado etc)
STATE_ABBV = 'co' # LOWERCASE state abbreviation (ca for California, co for Colorado etc)

WEEK = '' #1-52 (Defaults to middle of the year if '')

MAX_LINES = 10 # max number of train lines (leave as is unless expirementing with route generation)
# Training features
FEATURE_COLS = [
    'Month',
    'Week',
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
    'is_post_covid',
    'dist_to_nearest_job_center_km'
]

#Census Features
CENSUS_FEATURE_COLS = [
    'population_density',
    'low_car_rate',
    'transit_commuter_rate',
    'driver_rate',
    'carpool_rate',
    'employment_rate',
    'renter_rate',
    'median_income',
]

### Pulling ACS Census Data

In [43]:
vars_str = ','.join(CENSUS_VARIABLES)
county_str = '&in=county:' + ','.join(COUNTIES)

#pulling census data using requests
url = (
    f'https://api.census.gov/data/2023/acs/acs5'
    f'?get=NAME,{vars_str}'
    f'&for=tract:*'
    f'&in=state:{STATE_CODE}{county_str}'
    f'&key={CENSUS_KEY}'
)

print('Pulling ACS 2023 data for state tracts...')
response = requests.get(url)
data = response.json()
# Keeping only specified county data
state_tracts = pd.DataFrame(data[1:], columns=data[0])
state_tracts = state_tracts[state_tracts['county'].isin(COUNTIES)]
#Replace NaN equivalent number with NaN data
for col in CENSUS_VARIABLES:
    state_tracts[col] = pd.to_numeric(state_tracts[col], errors='coerce')
state_tracts.replace(-666666666, np.nan, inplace=True)
#Create unique tract ID
state_tracts['GEOID'] = state_tracts['state'] + state_tracts['county'] + state_tracts['tract']
print(f'Tracts loaded: {state_tracts.shape[0]}')
state_tracts.head()

Pulling ACS 2023 data for state tracts...
Tracts loaded: 694


,NAME,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E,state,county,tract,GEOID
0,Census Tract 78.01; Adams County; Colorado,4145,1332,602,375,1870,1276,230,365,3048,1870,1332,1150,44286.0,08,001,007801,08001007801
1,Census Tract 78.02; Adams County; Colorado,4092,1517,370,802,1868,1424,150,156,3102,1877,1517,1326,50646.0,08,001,007802,08001007802
2,Census Tract 79; Adams County; Colorado,6173,2264,288,700,3144,2514,420,98,4603,3164,2264,1262,67273.0,08,001,007900,08001007900
3,Census Tract 80; Adams County; Colorado,5701,2025,163,580,2813,2317,347,107,4545,2977,2025,983,77679.0,08,001,008000,08001008000
4,Census Tract 81; Adams County; Colorado,1615,770,95,426,783,288,5,102,1599,809,770,763,50000.0,08,001,008100,08001008100


### Load TIGER tract boundaries and join demographics

In [44]:
print('Downloading TIGER 2023 tract boundaries (CA)...')
tract_url = 'https://www2.census.gov/geo/tiger/TIGER2023/TRACT/tl_2023_' + STATE_CODE + '_tract.zip'

state_tract_geo = gpd.read_file(tract_url)
#Keep only tract boundaries within specified counties
state_tract_geo = state_tract_geo[state_tract_geo['COUNTYFP'].isin(COUNTIES)]
state_tract_geo = state_tract_geo.to_crs('EPSG:4326')
print(f'Tract geometries: {state_tract_geo.shape[0]}')
#Merge datasets with tract information and tract geometry
tracts = state_tract_geo[['GEOID', 'ALAND', 'geometry']].merge(
    state_tracts[['GEOID'] + CENSUS_VARIABLES],
    on='GEOID', how='left'
)
tracts.head()

Tract geometries: 694


,GEOID,ALAND,geometry,B01003_001E,B08201_001E,B08201_002E,B08201_003E,B08301_001E,B08301_002E,B08301_004E,B08301_010E,B23025_001E,B23025_004E,B25003_001E,B25003_003E,B19013_001E
0,08014030200,2583065,"POLYGON ((-105.09092 39.91435, -105.09082 39.9...",2244,979,43,376,1417,1120,272,78,1742,1439,979,538,102038.0
1,08005007103,165400308,"POLYGON ((-104.60375 39.64184, -104.60374 39.6...",1727,633,2,56,927,796,42,0,1371,934,633,26,141838.0
2,08005080500,1264541,"POLYGON ((-104.84727 39.6748, -104.84519 39.67...",3808,1766,38,919,1980,1395,148,103,3147,1985,1766,576,83013.0
3,08005083900,1376216,"POLYGON ((-104.8318 39.65308, -104.8316 39.653...",2901,1711,10,947,1637,1191,139,0,2604,1654,1711,600,72807.0
4,08013013703,179045632,"POLYGON ((-105.65184 40.02494, -105.64957 40.0...",1246,627,6,131,717,456,43,0,1053,730,627,179,84928.0


#### Engineer Model Features

In [45]:
#Add engineered features to tracts
def engineer_features(df):
    out = df.copy()
    out['population_density']    = out['B01003_001E'] / (out['ALAND'] / 1_000_000)
    out['low_car_rate']          = (out['B08201_002E'] + out['B08201_003E']) / out['B08201_001E']
    out['transit_commuter_rate'] = out['B08301_010E'] / out['B08301_001E']
    out['driver_rate']           = out['B08301_002E'] / out['B08301_001E']
    out['carpool_rate']          = out['B08301_004E'] / out['B08301_001E']
    out['employment_rate']       = out['B23025_004E'] / out['B23025_001E']
    out['renter_rate']           = out['B25003_003E'] / out['B25003_001E']
    out['median_income']         = out['B19013_001E']
    return out

tracts = engineer_features(tracts)
tracts[CENSUS_FEATURE_COLS].describe()

,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
count,694.000000,684.000000,684.000000,684.000000,684.000000,685.000000,684.000000,682.000000
mean,2184.550081,0.382403,0.030417,0.706010,0.071420,0.677872,0.367709,106191.885630
std,1828.452261,0.184051,0.033740,0.131271,0.048562,0.093699,0.248755,40711.245825
min,0.000000,0.000000,0.000000,0.185813,0.000000,0.000000,0.000000,22293.000000
25%,1095.774754,0.243544,0.007183,0.635816,0.036782,0.626285,0.154199,75412.000000
50%,1881.998913,0.354463,0.020088,0.728110,0.059197,0.684541,0.323910,99167.000000
75%,2760.943443,0.507361,0.044492,0.799247,0.092040,0.737482,0.539113,130238.500000
max,12764.584805,0.893660,0.331507,0.956149,0.278136,0.912345,1.000000,250001.000000


In [46]:
def dist_to_point(lat, lon, clat, clon):
    R = 6371
    dlat, dlon = np.radians(lat - clat), np.radians(lon - clon)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat))*np.cos(np.radians(clat))*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def dist_to_nearest_job_center(lat, lon, centers):
    dists = [dist_to_point(lat, lon, c[0], c[1]) for c in centers]
    return min(dists)


### Load Job-center Data (LODES)

In [47]:
lodes_url = 'https://lehd.ces.census.gov/data/lodes/LODES8/' + STATE_ABBV + '/wac/' + STATE_ABBV + '_wac_S000_JT00_2021.csv.gz'

lodes = pd.read_csv(lodes_url, compression='gzip', dtype={'w_geocode': str})

# w_geocode is an 11-digit census block — first 11 chars = tract GEOID
lodes['GEOID'] = lodes['w_geocode'].str[:11]

# Aggregate jobs to tract level
jobs_by_tract = lodes.groupby('GEOID')['C000'].sum().reset_index()
jobs_by_tract.rename(columns={'C000': 'total_jobs'}, inplace=True)

print(jobs_by_tract.sort_values('total_jobs', ascending=False).head(10))

           GEOID  total_jobs
447  08031001706       39560
171  08005006864       28272
160  08005006712       28093
448  08031001707       23875
445  08031001704       23798
293  08013012208       20862
166  08005006859       19000
713  08041002300       18739
546  08031006804       17799
932  08059010100       16580


In [48]:
#Merge job information into tract, calculate job density
tracts = tracts.merge(jobs_by_tract, on='GEOID', how='left')
tracts['job_density'] = tracts['total_jobs'] / (tracts['ALAND'] / 1_000_000)

#### Filter Job Centers

In [49]:
#Filter out tracts with job density above 95th percentile to get true job centers
emp_threshold = np.percentile(tracts['job_density'].dropna(), 95)
job_dense_tracts = tracts[tracts['job_density'] >= emp_threshold]
print(f'Tracts above 95th percentile of job density: {len(job_dense_tracts)}')

filtered_centers = []
#Keeep job centers that are within a certain distance to avoid redundant job centers
for _, row in job_dense_tracts.sort_values('job_density', ascending=False).iterrows():
    lat = row.geometry.centroid.y
    lon = row.geometry.centroid.x
    
    if not filtered_centers:
        filtered_centers.append((lat, lon))
        continue
    
    # compute distance to all already accepted centers
    dists = [
        dist_to_point(lat, lon, c[0], c[1])
        for c in filtered_centers
    ]
    
    if min(dists) >= MIN_JOB_KM:
        filtered_centers.append((lat, lon))

job_centers = np.array(filtered_centers)

print(f'Filtered job centers: {len(job_centers)}')
print(job_centers[:10])


Tracts above 95th percentile of job density: 35
Filtered job centers: 16
[[  39.74747811 -104.99090902]
 [  39.72966216 -104.98365225]
 [  39.61773634 -104.89193369]
 [  39.71936134 -104.95331632]
 [  40.0176125  -105.27915988]
 [  39.59904179 -104.90014187]
 [  39.7433727  -104.96658259]
 [  39.70426336 -104.93572583]
 [  39.67387055 -104.9359527 ]
 [  39.70382867 -104.98393158]]


### Generate Candidate Grid

In [50]:
def generate_grid(bounds, spacing_km):
    min_lon, min_lat, max_lon, max_lat = bounds
    deg_lat = spacing_km / 110.574
    deg_lon = spacing_km / (111.320 * np.cos(np.radians((min_lat + max_lat) / 2)))
    lats = np.arange(min_lat, max_lat, deg_lat)
    lons = np.arange(min_lon, max_lon, deg_lon)
    grid_lons, grid_lats = np.meshgrid(lons, lats)
    pts = pd.DataFrame({'lon': grid_lons.ravel(), 'lat': grid_lats.ravel()})
    return gpd.GeoDataFrame(
        pts,
        geometry=[Point(x, y) for x, y in zip(pts.lon, pts.lat)],
        crs='EPSG:4326'
    )

grid = generate_grid(BOUNDS, GRID_SPACING_KM)
print(f'Total grid points: {len(grid)}')

# Spatial join: each grid point inherits features of the tract it falls inside
grid_joined = gpd.sjoin(
    grid,
    tracts[['GEOID', 'ALAND', 'geometry'] + CENSUS_FEATURE_COLS],
    how='left',
    predicate='within'
)
# Drop points outside county boundary
grid_joined = grid_joined.dropna(subset=['GEOID']).reset_index(drop=True)
grid_joined['is_post_covid'] = 1
grid_joined['Week']  = WEEK if WEEK != '' else 24
grid_joined['Month'] = grid_joined['Week'].apply(lambda w: min(math.ceil(w / (52/12)), 12))
print(f'Candidates inside bounds: {len(grid_joined)}')
grid_joined['dist_to_nearest_job_center_km'] = grid_joined.apply(
    lambda r: dist_to_nearest_job_center(r.lat, r.lon, job_centers), axis=1
)

# Fill sparse NaNs with tract medians
for col in FEATURE_COLS:
    grid_joined[col] = grid_joined[col].fillna(grid_joined[col].median())

grid_joined[['lon','lat'] + FEATURE_COLS].head()

Total grid points: 19995
Candidates inside bounds: 15443


,lon,lat,Month,Week,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income,is_post_covid,dist_to_nearest_job_center_km
0,-105.350000,39.55,6,24,72.650239,0.207703,0.0,0.712558,0.076037,0.604457,0.086657,124655.0,1,25.817837
1,-105.344145,39.55,6,24,72.650239,0.207703,0.0,0.712558,0.076037,0.604457,0.086657,124655.0,1,25.468382
2,-105.338291,39.55,6,24,72.650239,0.207703,0.0,0.712558,0.076037,0.604457,0.086657,124655.0,1,25.124074
3,-105.332436,39.55,6,24,72.650239,0.207703,0.0,0.712558,0.076037,0.604457,0.086657,124655.0,1,24.785125
4,-105.326581,39.55,6,24,72.650239,0.207703,0.0,0.712558,0.076037,0.604457,0.086657,124655.0,1,24.451759


## Predict Using Model

In [51]:
model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

X = grid_joined[FEATURE_COLS].values
grid_joined['predicted_ridership'] = np.expm1(model.predict(X))

print(f"Score range: {grid_joined['predicted_ridership'].min():.0f} – {grid_joined['predicted_ridership'].max():.0f}")
grid_joined[['lon','lat','predicted_ridership']].sort_values('predicted_ridership', ascending=False).head(10)


Score range: 4636 – 27010


,lon,lat,predicted_ridership
12599,-105.262179,40.002186,27010.259766
12657,-105.268034,40.006708,26811.662109
12716,-105.268034,40.011230,26811.662109
12658,-105.262179,40.006708,26811.662109
5549,-104.998715,39.753484,24980.236328
5420,-104.998715,39.748962,24980.236328
5550,-104.992861,39.753484,24690.537109
5551,-104.987006,39.753484,24681.378906
5810,-104.981151,39.762527,24681.378906
5680,-104.987006,39.758005,24681.378906


## Greedy Station Selection

In [52]:
#Calculate distance between points
def haversine(lat1, lon1, lat2_arr, lon2_arr):
    R = 6371.0
    dlat = np.radians(lat2_arr - lat1)
    dlon = np.radians(lon2_arr - lon1)
    a = (np.sin(dlat/2)**2
         + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2_arr)) * np.sin(dlon/2)**2)
    return 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
#Select candidates within generated grid that have highest ridership
def greedy_select(candidates, n, min_spacing_km):
    ranked = candidates.sort_values('predicted_ridership', ascending=False)
    selected = []
    sel_lats, sel_lons = [], []

    for idx, row in ranked.iterrows():
        if len(selected) >= n:
            break
        if sel_lats:
            dists = haversine(row['lat'], row['lon'],
                              np.array(sel_lats), np.array(sel_lons))
            if dists.min() < min_spacing_km:
                continue
        selected.append(idx)
        sel_lats.append(row['lat'])
        sel_lons.append(row['lon'])

    return selected

selected_idx = greedy_select(grid_joined, N_STATIONS, MIN_SPACING_KM)
selected = grid_joined.loc[selected_idx].reset_index(drop=True)

print(f'Selected {len(selected)} stations')
print(f'Total predicted ridership: {selected["predicted_ridership"].sum():.0f}')
selected[['lat','lon','predicted_ridership'] + CENSUS_FEATURE_COLS].round(4)

Selected 15 stations
Total predicted ridership: 247521


,lat,lon,predicted_ridership,population_density,low_car_rate,transit_commuter_rate,driver_rate,carpool_rate,employment_rate,renter_rate,median_income
0,40.0022,-105.2622,27010.259766,4928.3754,0.8296,0.1011,0.1858,0.0138,0.2895,0.9611,33307.0
1,39.7535,-104.9987,24980.236328,5796.6102,0.7284,0.0941,0.2148,0.0000,0.5572,0.6261,92648.0
2,39.7128,-104.9753,16299.961914,8970.0683,0.6367,0.0066,0.5311,0.0033,0.8033,0.8052,111938.0
3,39.5952,-104.8992,16105.210938,3451.9434,0.7310,0.0749,0.6062,0.0625,0.8111,0.9927,79704.0
4,39.6540,-104.9812,16003.862305,3207.5537,0.7293,0.0468,0.5988,0.0630,0.5650,0.7774,51538.0
5,39.7399,-104.9636,15897.875000,6611.0232,0.6917,0.0788,0.6236,0.0315,0.7735,0.6921,76182.0
6,39.7264,-105.0163,15612.355469,627.1524,0.8787,0.3315,0.4438,0.1589,0.6063,0.9111,26462.0
7,39.6811,-104.9577,15518.829102,3600.4018,0.6596,0.0869,0.6507,0.1010,0.7056,0.8169,68301.0
8,39.9117,-104.9870,15189.993164,3645.7038,0.6271,0.0457,0.8572,0.0984,0.6685,0.8647,54701.0
9,39.6721,-104.8992,14968.876953,9313.2300,0.6034,0.0352,0.7310,0.1548,0.8743,0.9015,71538.0


### Route Generation (not necessary just for fun)

In [53]:
from scipy.optimize import linprog
import itertools
PERP_KM = 1.5

In [54]:
def straightness_score(lats, lons):
    if len(lats) < 2:
        return 0
    if len(np.unique(lons)) == 1:  # perfectly vertical line
        return 1.0
    if len(np.unique(lats)) == 1:  # perfectly horizontal line
        return 1.0
    # Try both orientations and take the best
    r2_a = LinearRegression().fit(lons.reshape(-1,1), lats).score(lons.reshape(-1,1), lats)
    r2_b = LinearRegression().fit(lats.reshape(-1,1), lons).score(lats.reshape(-1,1), lons)
    return max(r2_a, r2_b)

In [55]:
def generate_all_candidate_lines(selected, min_stations=3, perp_threshold_km=PERP_KM):
    """
    Generate every possible straight-ish line through subsets of stations.
    Returns list of (station_indices, score) tuples.
    """
    candidates = []
    n = len(selected)
    coords = selected[['lat', 'lon']].values
    ridership = selected['predicted_ridership'].values
    
    # For each pair of stations, define a line direction
    # then find all stations close to that line
    for i, j in itertools.combinations(range(n), 2):
        # Direction vector between stations i and j
        direction = coords[j] - coords[i]
        norm = np.linalg.norm(direction)
        if norm < 1e-9:
            continue
        direction = direction / norm
        origin = coords[i]
        lat_rad = np.radians(origin[0])
        
        # Find all stations within perp_threshold of this line
        on_line = []
        for k in range(n):
            v = coords[k] - origin
            proj = np.dot(v, direction) * direction
            perp = v - proj
            perp_km = np.sqrt(
                (perp[0] * 110.574)**2 +
                (perp[1] * 111.320 * np.cos(lat_rad))**2
            )
            if perp_km < perp_threshold_km:
                on_line.append(k)
        
        if len(on_line) < min_stations:
            continue
        
        # Score this candidate line
        subset_lats = coords[on_line, 0]
        subset_lons = coords[on_line, 1]
        r2 = straightness_score(subset_lats, subset_lons)
        total_ridership = ridership[on_line].sum()
        score = total_ridership * (r2 ** 2)  # square R² to penalize jaggedness harder
        
        candidates.append((on_line, score))
    
    # Deduplicate — same set of stations from different anchor pairs
    seen = set()
    unique = []
    for station_indices, score in candidates:
        key = frozenset(station_indices)
        if key not in seen:
            seen.add(key)
            unique.append((station_indices, score))
    
    return sorted(unique, key=lambda x: x[1], reverse=True)

def jaccard_similarity(set1, set2):
    intersection = len(set(set1) & set(set2))
    union = len(set(set1) | set(set2))
    return intersection / union

def select_lines_greedy(candidates, selected, max_lines=MAX_LINES,
                        interchange_percentile=60, max_similarity=0.7):
    """
    Greedily pick the best non-redundant lines.
    A line is redundant if >max_similarity% of its stations are already covered.
    High ridership stations can appear on multiple lines (interchanges).
    """
    interchange_threshold = np.percentile(
        selected['predicted_ridership'], interchange_percentile
    )
    
    covered = set()
    line_orders = {}
    selected_line_sets = []  # track station sets of chosen lines
    line_id = 0
    
    for station_indices, score in candidates:
        if line_id >= max_lines:
            break
        
        too_similar = any(
            jaccard_similarity(station_indices, existing) > max_similarity
            for existing in selected_line_sets
        )
        if too_similar:
            continue
        
        # Skip if too many non-interchange stations already covered
        non_interchange_covered = [
            i for i in station_indices
            if i in covered and 
            selected.iloc[i]['predicted_ridership'] < interchange_threshold
        ]
        if len(non_interchange_covered) / len(station_indices) > 0.5:
            continue
        
         # Order stations along the line
        coords = selected[['lat', 'lon']].values
        subset_coords = coords[station_indices]
        centered = subset_coords - subset_coords.mean(axis=0)
        _, _, Vt = np.linalg.svd(centered)
        direction = Vt[0]
        projections = centered @ direction
        order = np.argsort(projections)
        ordered_indices = [station_indices[i] for i in order]
        df_indices = selected.iloc[ordered_indices].index.tolist()
        
        line_orders[line_id] = df_indices
        selected_line_sets.append(set(station_indices))
        
        # Mark non-interchange stations as covered
        for i in station_indices:
            if selected.iloc[i]['predicted_ridership'] < interchange_threshold:
                covered.add(i)
        
        r2 = straightness_score(
            selected.iloc[ordered_indices]['lat'].values,
            selected.iloc[ordered_indices]['lon'].values
        )
        ridership = selected.iloc[ordered_indices]['predicted_ridership'].sum()
        print(f'Line {line_id+1}: {len(ordered_indices)} stations | '
              f'R²={r2:.3f} | ridership={ridership:.0f} | score={score:.0f}')
        
        line_id += 1
    
    return line_orders

In [56]:
print('Generating candidate lines...')
candidates = generate_all_candidate_lines(
    selected, 
    min_stations=3, 
    perp_threshold_km=PERP_KM
)
print(f'Unique candidate lines: {len(candidates)}')

print('\nSelecting best lines...')
line_orders = select_lines_greedy(
    candidates, selected, MAX_LINES
)

print(f'\nTotal lines selected: {len(line_orders)}')

for line_id, station_idx in line_orders.items():
    print(f'\nLine {line_id + 1}:')
    print(selected.loc[station_idx][['lat', 'lon', 'predicted_ridership']].to_string())

Generating candidate lines...
Unique candidate lines: 42

Selecting best lines...
Line 1: 5 stations | R²=0.988 | ridership=99157 | score=96801
Line 2: 5 stations | R²=0.993 | ridership=97959 | score=96612
Line 3: 5 stations | R²=0.995 | ridership=88591 | score=87637
Line 4: 5 stations | R²=0.992 | ridership=88311 | score=86901
Line 5: 5 stations | R²=0.975 | ridership=87054 | score=82746
Line 6: 4 stations | R²=0.979 | ridership=68074 | score=65217
Line 7: 3 stations | R²=0.999 | ridership=56594 | score=56445
Line 8: 3 stations | R²=0.961 | ridership=55847 | score=51558
Line 9: 3 stations | R²=0.944 | ridership=53508 | score=47704
Line 10: 3 stations | R²=1.000 | ridership=45445 | score=45443

Total lines selected: 10

Line 1:
         lat         lon  predicted_ridership
0  40.002186 -105.262179         27010.259766
1  39.753484 -104.998715         24980.236328
5  39.739918 -104.963587         15897.875000
2  39.712787 -104.975296         16299.961914
9  39.672090 -104.899185        

## MAPPING

In [59]:
cmap = plt.cm.YlOrRd
vmin = selected['predicted_ridership'].min()
vmax = selected['predicted_ridership'].max()

def score_to_hex(val):
    norm = (val - vmin) / (vmax - vmin + 1e-9)
    r, g, b, _ = cmap(norm)
    return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

m = folium.Map(location=[BOUND_AVG[1], BOUND_AVG[0]], zoom_start=11, tiles='cartodbpositron')

# Background heatmap of all candidate scores
heat_data = grid_joined[['lat','lon','predicted_ridership']].values.tolist()
HeatMap(heat_data, radius=14, blur=18, min_opacity=0.3,
        gradient={'0.3':'blue','0.55':'lime','0.75':'orange','1.0':'red'}).add_to(m)

# Selected station markers
for i, row in selected.iterrows():
    color = score_to_hex(row['predicted_ridership'])
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=10,
        color='white', weight=2,
        fill=True, fill_color=color, fill_opacity=0.95,
        tooltip=(
            f"<b>Station {i+1}</b><br>"
            f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
            f"Pop density: {row['population_density']:.0f}/km²<br>"
            f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
            f"Low-car rate: {row['low_car_rate']:.1%}<br>"
            f"Employment rate: {row['employment_rate']:.1%}<br>"
            f"Renter rate: {row['renter_rate']:.1%}<br>"
            f"Median income: ${row['median_income']:,.0f}"
        )
    ).add_to(m)

legend_html = """
<div style='position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:12px 16px;border-radius:8px;
     box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px'>
  <b>Predicted Ridership</b><br>
  <span style='color:#d73027'>●</span> High &nbsp;
  <span style='color:#fee08b'>●</span> Medium &nbsp;
  <span style='color:#4575b4'>●</span> Low<br><br>
  <i>Hover markers for tract features</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.save('transit_stations.html')
print('Saved → transit_stations.html')

Saved → transit_stations.html


### Mapping with Routes Generated (not necessary just for fun)

In [58]:
# LINE_COLORS = [
#     '#e41a1c','#377eb8','#4daf4a','#ff7f00',
#     '#984ea3','#a65628','#f781bf','#999999',
#     '#00ced1','#ffd700'
# ]

# cmap = plt.cm.YlOrRd
# vmin = selected['predicted_ridership'].min()
# vmax = selected['predicted_ridership'].max()

# def score_to_hex(val):
#     norm = (val - vmin) / (vmax - vmin + 1e-9)
#     r, g, b, _ = cmap(norm)
#     return '#{:02x}{:02x}{:02x}'.format(int(r*255), int(g*255), int(b*255))

# m = folium.Map(location=[BOUND_AVG[1], BOUND_AVG[0]], zoom_start=11, tiles='CartoDB positron')

# # Background heatmap
# heat_data = grid_joined[['lat','lon','predicted_ridership']].values.tolist()
# HeatMap(heat_data, radius=14, blur=18, min_opacity=0.3,
#         gradient={'0.3':'blue','0.55':'lime','0.75':'orange','1.0':'red'}).add_to(m)

# # # Draw lines first
# # for line_id, station_idx in line_orders.items():
# #     color = LINE_COLORS[line_id % len(LINE_COLORS)]
# #     line_stations = selected.loc[station_idx]
# #     coords = [[row['lat'], row['lon']] for _, row in line_stations.iterrows()]
# #     folium.PolyLine(
# #         coords, color=color, weight=5, opacity=0.8,
# #         tooltip=f'Line {line_id + 1}'
# #     ).add_to(m)

# # All selected stations — grey if not on any line
# assigned_idx = set(idx for idxs in line_orders.values() for idx in idxs)

# for i, row in selected.iterrows():
#     if i not in assigned_idx:
#         folium.CircleMarker(
#             location=[row['lat'], row['lon']],
#             radius=7,
#             color='white', weight=2,
#             fill=True, fill_color='#888888', fill_opacity=0.7,
#             tooltip=(
#                 f"<b>Unassigned Station</b><br>"
#                 f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
#                 f"Pop density: {row['population_density']:.0f}/km²<br>"
#                 f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
#                 f"Low-car rate: {row['low_car_rate']:.1%}<br>"
#                 f"Employment rate: {row['employment_rate']:.1%}<br>"
#                 f"Renter rate: {row['renter_rate']:.1%}<br>"
#                 f"Median income: ${row['median_income']:,.0f}<br>"
#                 f"Job proximity: {row['dist_to_nearest_job_center_km']:.1f} km"
#             )
#         ).add_to(m)

# # Station markers colored by line
# for line_id, station_idx in line_orders.items():
#     color = LINE_COLORS[line_id % len(LINE_COLORS)]
#     line_stations = selected.loc[station_idx]
#     for i, (idx, row) in enumerate(line_stations.iterrows()):
#         is_terminal = (i == 0 or i == len(line_stations) - 1)
#         is_interchange = selected.loc[idx, 'is_interchange'] if 'is_interchange' in selected.columns else False
#         folium.CircleMarker(
#             location=[row['lat'], row['lon']],
#             radius=12 if is_interchange else (10 if is_terminal else 7),
#             color='white', weight=2,
#             fill=True, fill_color=color, fill_opacity=0.95,
#             tooltip=(
#                 f"<b>Line {line_id+1} — Station {i+1}</b><br>"
#                 f"{'🔀 INTERCHANGE — ' if is_interchange else ''}"
#                 f"{'⬤ TERMINAL — ' if is_terminal else ''}"
#                 f"Predicted ridership: <b>{row['predicted_ridership']:.0f}</b><br>"
#                 f"Pop density: {row['population_density']:.0f}/km²<br>"
#                 f"Transit rate: {row['transit_commuter_rate']:.1%}<br>"
#                 f"Low-car rate: {row['low_car_rate']:.1%}<br>"
#                 f"Employment rate: {row['employment_rate']:.1%}<br>"
#                 f"Renter rate: {row['renter_rate']:.1%}<br>"
#                 f"Median income: ${row['median_income']:,.0f}<br>"
#                 f"Job proximity: {row['dist_to_nearest_job_center_km']:.1f} km"
#             )
#         ).add_to(m)

# # Legend
# legend_items = ''.join([
#     f"<span style='color:{LINE_COLORS[i % len(LINE_COLORS)]}'>━━</span> Line {i+1}&nbsp;&nbsp;"
#     for i in range(len(line_orders))
# ])
# legend_html = f"""
# <div style='position:fixed;bottom:30px;left:30px;z-index:1000;
#      background:white;padding:12px 16px;border-radius:8px;
#      box-shadow:2px 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:13px;max-width:400px'>
#   <b>Transit Lines</b><br>{legend_items}<br><br>
#   <span style='font-size:11px'>
#   ⬤ Terminal &nbsp;|&nbsp; 🔀 Interchange &nbsp;|&nbsp; ⬤ Unassigned &nbsp;|&nbsp; Hover for details
#   </span>
# </div>
# """
# m.get_root().html.add_child(folium.Element(legend_html))

# m.save('transit_stations.html')
# print('Saved → transit_stations.html')
# m